# Appendix A — Field Guide to Indian Data Sources

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/appA_indian_data_sources.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## A.1 What the book's cached data cover

In [ ]:
from pathlib import Path

import pandas as pd

RAW = Path("data/raw")
MONTHS = {m: i for i, m in enumerate(
    ["January", "February", "March", "April", "May", "June", "July",
     "August", "September", "October", "November", "December"], start=1)}


def periods(df: pd.DataFrame, kind: str) -> pd.Series:
    """Turn the period columns of a raw file into sortable labels."""
    if kind == "date":
        return pd.to_datetime(df["date"]).dt.strftime("%Y-%m")
    if kind == "year-month":
        return (df["year"].astype(int).astype(str) + "-"
                + df["month"].map(MONTHS).map("{:02d}".format))
    if kind == "fiscal":                      # "2023-24" sorts correctly as text
        return df["year"].astype(str)
    return df["year"].astype(str)             # calendar year


SOURCES = [   # (label, file pattern, columns to read, kind of period)
    ("CPI general, combined (mospi_cpi)", "mospi_cpi_combined_99_0.csv",
     ["date", "fetched_on"], "date"),
    ("CPI groups (mospi_cpi_groups)", "mospi_cpi_getCPIIndex_*state_code-99.csv",
     ["year", "month", "fetched_on"], "year-month"),
    ("CPI items (mospi_cpi_items)", "mospi_cpi_getItemIndex_year-*.csv",
     ["year", "month", "fetched_on"], "year-month"),
    ("WPI, all levels (mospi_wpi)", "mospi_wpi_getWpiRecords_year-*.csv",
     ["year", "month", "fetched_on"], "year-month"),
    ("IIP monthly, base 2011-12", "mospi_iip_getIIPMonthly_base_year-2011-12.csv",
     ["year", "month", "fetched_on"], "year-month"),
    ("IIP monthly, base 2022-23", "mospi_iip_getIIPMonthly_base_year-2022-23.csv",
     ["year", "month", "fetched_on"], "year-month"),
    ("NAS annual GDP, base 2011-12",
     "mospi_nas_*2011-12_frequency_code-1_indicator_code-5_*.csv",
     ["year", "fetched_on"], "fiscal"),
    ("NAS annual GDP, base 2022-23",
     "mospi_nas_*2022-23_frequency_code-1_indicator_code-5_*.csv",
     ["year", "fetched_on"], "fiscal"),
    ("NAS state GSDP (code 23)", "mospi_nas_*frequency_code-1_indicator_code-23_*.csv",
     ["year", "fetched_on"], "fiscal"),
    ("BIS policy rate (bis WS_CBPOL)", "bis_WS_CBPOL_M.IN.csv",
     ["date", "fetched_on"], "date"),
    ("BIS rupee per US dollar (WS_XRU)", "bis_WS_XRU_M.IN.INR.A.csv",
     ["date", "fetched_on"], "date"),
    ("BIS REER, broad (WS_EER)", "bis_WS_EER_M.R.B.IN.csv",
     ["date", "fetched_on"], "date"),
    ("OECD interest rates (oecd_rates)", "oecd_rates_IND.csv",
     ["date", "fetched_on"], "date"),
    ("World Bank GDP growth (worldbank)", "wb_NY.GDP.MKTP.KD.ZG_IND.csv",
     ["year", "fetched_on"], "year"),
]

rows = []
for label, pattern, cols, kind in SOURCES:
    files = sorted(RAW.glob(pattern))
    if not files:
        rows.append({"source": label, "files": 0})
        continue
    df = pd.concat([pd.read_csv(f, usecols=cols, dtype=str) for f in files])
    p = periods(df, kind)
    rows.append({"source": label, "files": len(files), "rows": len(df),
                 "first": p.min(), "last": p.max(),
                 "fetched_on": df["fetched_on"].max()})
coverage = pd.DataFrame(rows).set_index("source")
coverage

In [ ]:
pwt = pd.read_stata(RAW / "pwt1001.dta", columns=["countrycode", "year"])
print(f"Penn World Table 10.01: {pwt['countrycode'].nunique()} countries, "
      f"{pwt['year'].min()}-{pwt['year'].max()}")
del pwt
print("DoCA price files:", [f.name for f in sorted(RAW.glob("doca_*.csv"))])
print("Union Budget PDFs:", len(list((RAW / "budget").glob("*.pdf"))), "files")

## A.2 MoSPI eSankhyiki: the API

In [ ]:
from indiadata import NAS_INDICATORS, NAS_REVISIONS


def nas_unit(code: int) -> str:
    """Unit and years of an annual NAS table, from its cached file."""
    found = list(RAW.glob(f"mospi_nas_*2011-12_frequency_code-1_"
                          f"indicator_code-{code}_*.csv"))
    if not found:
        return "(not cached)"
    df = pd.read_csv(found[0], usecols=["year", "unit"], dtype=str)
    return f"{df['unit'].iloc[0]}, {df['year'].min()} to {df['year'].max()}"


codes = pd.DataFrame({"name": list(NAS_INDICATORS),
                      "code": list(NAS_INDICATORS.values())})
codes["unit and years (cached)"] = codes["code"].map(nas_unit)
codes.set_index("code").sort_index()

In [ ]:
print(" -> ".join(NAS_REVISIONS))

### Quirks of MoSPI API data

## A.3 MoSPI Microdata Portal: unit-level surveys

### Quirks of PLFS unit-level data

## A.4 Reserve Bank of India

## A.5 Union Budget documents

## A.6 International sources

## A.7 Other sources used in the book

## A.8 A checklist for any new source